- treina o XGBoost e a Random Forest com todos os dados disponíveis (sem separar 2025 para teste)

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import itertools
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

# dataframe vindo do notebook anterior
caminho_arquivo = 'nfl_features_ml_2021_2025.parquet'

try:
    df_ml = pd.read_parquet(caminho_arquivo)
    print(f"Dataset carregado com sucesso! Total de jogos: {len(df_ml)}")
except Exception as e:
    print("Tentando carregar CSV como fallback...")
    df_ml = pd.read_csv('nfl_features_ml_2021_2025.csv')

Dataset carregado com sucesso! Total de jogos: 1339


In [2]:
# 1. As Features Campeãs do seu GridSearch Manual
features_rf = [
    # Pts (Ambos: EWMA/Last3 + Season)
        'home_offense_pts_last3', 'home_defense_pts_last3', 
        'home_offense_pts_season', 'home_defense_pts_season',
        'away_offense_pts_last3', 'away_defense_pts_last3', 
        'away_offense_pts_season', 'away_defense_pts_season',
        
        # Turnovers (Apenas o Momento Recente / EWMA)
        'home_turnovers_last3', 
        'away_turnovers_last3'
]

features_xgb = [
    # Pts (Ambos: EWMA/Last3 + Season)
    'home_offense_pts_last3', 'home_defense_pts_last3', 
    'home_offense_pts_season', 'home_defense_pts_season',
    'away_offense_pts_last3', 'away_defense_pts_last3', 
    'away_offense_pts_season', 'away_defense_pts_season',
    
    # Turnovers (Ambos: EWMA/Last3 + Season)
    'home_turnovers_last3', 'home_turnovers_season',
    'away_turnovers_last3', 'away_turnovers_season'
]

In [3]:
# 2. Treinando os Modelos "Gold" respeitando a dieta de cada um
xgb_gold = XGBClassifier(
    n_estimators=100, 
    max_depth=4, 
    learning_rate=0.01, 
    subsample=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb_gold.fit(df_ml[features_xgb], df_ml['target'])

rf_gold = RandomForestClassifier(
    max_depth=7,
    max_features='sqrt',
    min_samples_leaf=2,
    n_estimators=100,
    random_state=42
)
rf_gold.fit(df_ml[features_rf], df_ml['target'])

print("Motores treinados com sucesso com features independentes!")

Motores treinados com sucesso com features independentes!


In [4]:
def pegar_ultimas_estatisticas(time, df_historico):
    """
    Busca o último jogo de um time no dataset e extrai suas estatísticas finais,
    garantindo que puxa os turnovers da temporada E recentes.
    """
    jogos_time = df_historico[(df_historico['home_team'] == time) | (df_historico['away_team'] == time)].copy()
    
    if jogos_time.empty:
        raise ValueError(f"Time {time} não encontrado no dataset!")
        
    ultimo_jogo = jogos_time.iloc[-1]
    
    stats = {}
    if ultimo_jogo['home_team'] == time:
        stats['offense_pts_last3'] = ultimo_jogo['home_offense_pts_last3']
        stats['defense_pts_last3'] = ultimo_jogo['home_defense_pts_last3']
        stats['offense_pts_season'] = ultimo_jogo['home_offense_pts_season']
        stats['defense_pts_season'] = ultimo_jogo['home_defense_pts_season']
        stats['turnovers_last3'] = ultimo_jogo['home_turnovers_last3']
        stats['turnovers_season'] = ultimo_jogo['home_turnovers_season']
    else:
        stats['offense_pts_last3'] = ultimo_jogo['away_offense_pts_last3']
        stats['defense_pts_last3'] = ultimo_jogo['away_defense_pts_last3']
        stats['offense_pts_season'] = ultimo_jogo['away_offense_pts_season']
        stats['defense_pts_season'] = ultimo_jogo['away_defense_pts_season']
        stats['turnovers_last3'] = ultimo_jogo['away_turnovers_last3']
        stats['turnovers_season'] = ultimo_jogo['away_turnovers_season']
        
    return stats

In [6]:
def prever_jogo(mandante, visitante, df_historico, features_xgb, features_rf):
    # 1. Puxa as estatísticas mais recentes
    stats_mandante = pegar_ultimas_estatisticas(mandante, df_historico)
    stats_visitante = pegar_ultimas_estatisticas(visitante, df_historico)
    
    # 2. Monta o dicionário mestre com TODAS as variáveis
    jogo_futuro = {
        'home_offense_pts_last3': stats_mandante['offense_pts_last3'],
        'home_defense_pts_last3': stats_mandante['defense_pts_last3'],
        'away_offense_pts_last3': stats_visitante['offense_pts_last3'],
        'away_defense_pts_last3': stats_visitante['defense_pts_last3'],
        
        'home_offense_pts_season': stats_mandante['offense_pts_season'],
        'home_defense_pts_season': stats_mandante['defense_pts_season'],
        'away_offense_pts_season': stats_visitante['offense_pts_season'],
        'away_defense_pts_season': stats_visitante['defense_pts_season'],
        
        'home_turnovers_last3': stats_mandante['turnovers_last3'],
        'home_turnovers_season': stats_mandante['turnovers_season'],
        'away_turnovers_last3': stats_visitante['turnovers_last3'],
        'away_turnovers_season': stats_visitante['turnovers_season']
    }
    
    # 3. Separa os DataFrames exatamente como cada modelo exige
    df_mestre = pd.DataFrame([jogo_futuro])
    df_predict_xgb = df_mestre[features_xgb]
    df_predict_rf = df_mestre[features_rf]
    
    # 4. Inferência individualizada
    prob_xgb = xgb_gold.predict_proba(df_predict_xgb)[0][1]
    prob_rf = rf_gold.predict_proba(df_predict_rf)[0][1]
    
    # 5. Avalia o Comitê e o Toss-Up
    media_prob = (prob_xgb + prob_rf) / 2
    status = "VÁLIDO PARA APOSTA" if (media_prob > 0.55 or media_prob < 0.45) else "TOSS-UP (IGNORAR)"
    
    favorito = mandante if media_prob >= 0.50 else visitante
    confianca = media_prob if media_prob >= 0.50 else (1 - media_prob)
    
    print(f"🏈 {mandante} (Mandante) vs {visitante} (Visitante) 🏈")
    print("-" * 40)
    print(f"Prob. XGBoost: {prob_xgb * 100:.1f}% para {mandante}")
    print(f"Prob. RF     : {prob_rf * 100:.1f}% para {mandante}")
    print("-" * 40)
    print(f"🔥 Veredito do Comitê:")
    print(f"Favorito: {favorito} (Confiança: {confianca * 100:.1f}%)")
    print(f"Status do Filtro: {status}\n")


In [13]:
# === TESTANDO O MOTOR ===
# As duas listas precisam ser passadas agora
prever_jogo('HOU', 'BUF', df_ml, features_xgb, features_rf)

🏈 HOU (Mandante) vs BUF (Visitante) 🏈
----------------------------------------
Prob. XGBoost: 48.8% para HOU
Prob. RF     : 57.8% para HOU
----------------------------------------
🔥 Veredito do Comitê:
Favorito: HOU (Confiança: 53.3%)
Status do Filtro: TOSS-UP (IGNORAR)

